# Light-weight ViTs — ImageNet-100 training notebook

Train each of the **14 hand-built models** (3 MobileViT + 6 EfficientViT + 5 EfficientFormer)
on **ImageNet-100** through the unified pipeline. The dataset and dataloaders are
built **once** at the top; every model then trains in its own cell via a single
`run(...)` call, reporting **top-1 and top-5** accuracy and saving only the best
weights to `best_<model>.pt`.

### Practical approach (why one `run()` + one cell per model)
Rather than copy-pasting a full training loop into 14 cells, the setup defines a
single `run(model_name, **overrides)` function and shares the dataloaders. Each
model cell is then a one-liner you can run, re-run, or skip independently — if a
cell crashes (e.g. OOM) the others and the shared data are untouched.

### Logging & early stopping
- **Per-model log file:** every `run(...)` writes `logs/<model>.log` (one line
  per epoch) **in addition** to the inline output, so each case has its own log.
- **Early stopping:** training stops if validation top-1 does not improve for
  `EARLY_STOPPING_PATIENCE` epochs (default **10**; set 0 to disable).

### A100 guidance (rough, single 40/80 GB A100, AMP on, 224px)
ImageNet-100 has ~126k train / ~5k val images. These models are small, so the
GPU is often **data-loading bound** — use enough `num_workers` and fast storage.

| Setting        | Recommendation |
|----------------|----------------|
| Image size     | **224** (required for EfficientViT/EfficientFormer; fine for MobileViT) |
| Batch size     | **256** (drop to 128 for `efficientformer_l7` if you OOM) |
| Epochs         | **100** for a solid benchmark (use 300 to chase SOTA; 1–2 to smoke-test) |
| Optimizer      | AdamW, lr 1e-3, wd 0.05, **5-epoch warmup**, cosine decay, label smoothing 0.1 |
| AMP            | on |
| Early stopping | patience **10** on val top-1 (`EARLY_STOPPING_PATIENCE`) |

**Approx. wall-clock per model @ 100 epochs** (compute + typical data overhead):

| Model group                                   | ~ time / epoch | ~ 100 epochs |
|-----------------------------------------------|----------------|--------------|
| small (mobilevit_xxs/xs, efficientvit_m0-m2, efficientformer_l1/l3_mini) | ~0.7–1.2 min | **~1.5–2 h** |
| medium (mobilevit_s, efficientvit_m3-m5, efficientformer_l3/l7_mini)     | ~1.2–2 min   | **~2–3.5 h** |
| large (efficientformer_l7)                                               | ~2–3 min     | **~3.5–5 h** |

So the **14 models at 100 epochs ≈ 30–45 GPU-hours total** on one A100 (≈ 1.5–2
days unattended) — less with early stopping. At 300 epochs multiply by ~3. If you
only need a quick ranking, **30–50 epochs** already separates the models. (These
are estimates; the printed `img/s` in the first epoch lets you extrapolate.)


## 1. Download ImageNet-100

**Where:** the most convenient public copy is on Kaggle —
`ambityga/imagenet100` (<https://www.kaggle.com/datasets/ambityga/imagenet100>),
~16 GB, 100 classes, ~1300 train + 50 val images per class. It unzips to
`train.X1 .. train.X4` + `val.X` folders (plus `Labels.json`), which the prep
cell below consolidates into the standard `train/` and `val/` ImageFolder layout.

**How (pick one):**
- **Kaggle notebook:** click *Add Data → ambityga/imagenet100*; it appears at
  `/kaggle/input/imagenet100` (read-only). Set `SRC_DIR` to that path.
- **Colab / cloud VM:** use the Kaggle API. Upload your `kaggle.json` token
  (Kaggle → Account → *Create New API Token*) so it lives at `~/.kaggle/kaggle.json`,
  then the download cell fetches and unzips it.
- **Already have ImageNet-100** in `train/`+`val/` form? Just point `DATA_ROOT`
  at it and skip the download cell.

Alternative subsets exist (e.g. the CMC class list applied to full ImageNet); any
`train/<class>/*` + `val/<class>/*` tree works.

In [ ]:
# --- 2. Environment & repo path -------------------------------------------------
import os, sys, subprocess
from pathlib import Path

# Point this at the "Light-wight ViTs" package directory (the folder that
# contains models/, data/, training/, inference/). Override via env var if needed.
REPO_DIR = Path(os.environ.get("LWVIT_REPO", Path.cwd()))
if REPO_DIR.name == "notebooks":          # running from inside notebooks/
    REPO_DIR = REPO_DIR.parent
# Common clone locations as fallbacks:
for cand in [REPO_DIR, Path("/content/Light-wight ViTs"), Path("Light-wight ViTs")]:
    if (cand / "models").is_dir():
        REPO_DIR = cand
        break
sys.path.insert(0, str(REPO_DIR))
print("REPO_DIR =", REPO_DIR)

# Dependencies (usually preinstalled on Colab/Kaggle). Uncomment if missing:
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "pillow"])

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Sanity import of the library.
from models import list_models, create_model, count_parameters_millions
print("registered models:", len(list_models()))


In [ ]:
# --- 3. Configuration -----------------------------------------------------------
# Edit these to taste. DATA_ROOT is where the FINAL train/ + val/ live.
SRC_DIR    = Path(os.environ.get("LWVIT_SRC", "/kaggle/input/imagenet100"))  # raw/extracted source (Kaggle input or download dir)
DATA_ROOT  = Path(os.environ.get("LWVIT_DATA", "./imagenet100"))             # consolidated ImageFolder root used for training
CKPT_DIR   = Path("./checkpoints"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR    = Path("./logs");        LOG_DIR.mkdir(parents=True, exist_ok=True)   # per-model train logs

IMG_SIZE        = 224     # 224 is required by EfficientViT/EfficientFormer
BATCH_SIZE      = 256     # lower to 128 if efficientformer_l7 OOMs
EPOCHS          = 100     # set 1–2 for a quick smoke test (see QUICK_TEST below)
NUM_WORKERS     = 8
AMP             = torch.cuda.is_available()

LR              = 1e-3
WEIGHT_DECAY    = 0.05
OPTIMIZER       = "adamw"
SCHEDULER       = "cosine"
WARMUP_EPOCHS   = 5
MIN_LR          = 1e-5
LABEL_SMOOTHING = 0.1
MONITOR         = "top1"   # best.pt is saved on best validation top-1
GRAD_CLIP_NORM  = 1.0
EARLY_STOPPING_PATIENCE = 10   # stop if val top-1 stalls for N epochs (0 disables)

# QUICK_TEST trains on a small Subset for a couple of epochs to validate the whole
# pipeline end-to-end in minutes before committing to full runs.
QUICK_TEST          = False
QUICK_TEST_EPOCHS   = 2
QUICK_TEST_SUBSET   = 2000   # images per split when QUICK_TEST is on

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DATA_ROOT={DATA_ROOT} | img={IMG_SIZE} | batch={BATCH_SIZE} | epochs={EPOCHS} | "
      f"amp={AMP} | early_stop={EARLY_STOPPING_PATIENCE} | device={DEVICE}")
print(f"logs -> {LOG_DIR}/<model>.log   checkpoints -> {CKPT_DIR}/best_<model>.pt")


In [ ]:
# --- 4. Download + consolidate ImageNet-100 into train/ + val/ ------------------
import glob, zipfile, shutil

def _have_imagefolder(root: Path) -> bool:
    return (root / "train").is_dir() and (root / "val").is_dir()

def _kaggle_download(dst: Path) -> Path:
    """Download + unzip ambityga/imagenet100 via the Kaggle API into dst."""
    dst.mkdir(parents=True, exist_ok=True)
    print("Downloading ambityga/imagenet100 via Kaggle API ...")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", "ambityga/imagenet100", "-p", str(dst)],
        check=True,
    )
    for z in dst.glob("*.zip"):
        print("Unzipping", z.name, "...")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dst)
    return dst

def _link_or_move(src: Path, dst: Path):
    """Prefer a symlink (no copy); fall back to move if symlinks are unavailable."""
    if dst.exists():
        return
    try:
        dst.symlink_to(src.resolve(), target_is_directory=True)
    except Exception:
        shutil.move(str(src), str(dst))

def consolidate(src: Path, dst: Path) -> Path:
    """Build dst/train + dst/val from src's train.X*/val.X (or train/val)."""
    (dst / "train").mkdir(parents=True, exist_ok=True)
    (dst / "val").mkdir(parents=True, exist_ok=True)
    train_srcs = sorted(src.glob("train.X*")) or ([src / "train"] if (src / "train").is_dir() else [])
    val_srcs   = sorted(src.glob("val.X*"))   or ([src / "val"]   if (src / "val").is_dir()   else [])
    if not train_srcs or not val_srcs:
        raise FileNotFoundError(
            f"Could not find train/val splits under {src}. Expected train.X*/val.X "
            "(Kaggle ambityga/imagenet100) or train/ + val/."
        )
    for group in train_srcs:
        for cls in sorted(p for p in group.iterdir() if p.is_dir()):
            _link_or_move(cls, dst / "train" / cls.name)
    for group in val_srcs:
        for cls in sorted(p for p in group.iterdir() if p.is_dir()):
            _link_or_move(cls, dst / "val" / cls.name)
    return dst

# Resolve the dataset: use DATA_ROOT if already prepared; else consolidate from
# SRC_DIR; else download via the Kaggle API into ./_imagenet100_raw and consolidate.
if _have_imagefolder(DATA_ROOT):
    print("Found prepared ImageFolder at", DATA_ROOT)
elif SRC_DIR.exists():
    print("Consolidating from", SRC_DIR, "->", DATA_ROOT)
    consolidate(SRC_DIR, DATA_ROOT)
else:
    raw = _kaggle_download(Path("./_imagenet100_raw"))
    consolidate(raw, DATA_ROOT)

assert _have_imagefolder(DATA_ROOT), f"ImageFolder not ready at {DATA_ROOT}"
n_train_classes = len(list((DATA_ROOT / "train").iterdir()))
print("train classes:", n_train_classes)


In [ ]:
# --- 5. Build the dataset + dataloaders ONCE (shared by every model) -----------
from torch.utils.data import Subset
from data import (
    build_imagenet100k_datasets, build_fixed_scale_loader,
    make_train_transform_builder, make_eval_transform, IMAGENET_MEAN, IMAGENET_STD,
)

train_ds, val_ds = build_imagenet100k_datasets(DATA_ROOT)
NUM_CLASSES = len(train_ds.classes)
CLASS_NAMES = train_ds.classes
print(f"ImageNet-100 | train {len(train_ds)} | val {len(val_ds)} | classes {NUM_CLASSES}")

# Optional quick-test subsetting for fast end-to-end validation.
_train_src, _val_src = train_ds, val_ds
_epochs = EPOCHS
if QUICK_TEST:
    import torch as _t
    _train_src = Subset(train_ds, _t.randperm(len(train_ds))[:QUICK_TEST_SUBSET].tolist())
    _val_src   = Subset(val_ds,   _t.randperm(len(val_ds))[:QUICK_TEST_SUBSET].tolist())
    _epochs = QUICK_TEST_EPOCHS
    print(f"QUICK_TEST on: {len(_train_src)} train / {len(_val_src)} val, {_epochs} epochs")

# Train transform = ImageNet RandomResizedCrop; eval = resize+center-crop.
train_loader = build_fixed_scale_loader(
    _train_src,
    transform=make_train_transform_builder(mean=IMAGENET_MEAN, std=IMAGENET_STD)(IMG_SIZE),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True,
)
val_loader = build_fixed_scale_loader(
    _val_src,
    transform=make_eval_transform(IMG_SIZE, mean=IMAGENET_MEAN, std=IMAGENET_STD),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
)
print("steps/epoch:", len(train_loader), "| val batches:", len(val_loader))


In [ ]:
# --- 6. The single training function used by every model cell ------------------
import logging, time
from torch import nn
from models import create_model, count_parameters_millions, get_model_info
from training import Trainer, build_optimizer, build_scheduler

# Show the trainer's per-epoch logs inline in the notebook (stdout).
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
_LOG_FMT = logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S")

RESULTS = {}   # model_name -> dict of summary metrics

def run(model_name: str, epochs: int = None, **overrides):
    """Train one model on the shared ImageNet-100 loaders; report top-1 & top-5.

    All 14 models share the loaders above (224px). Fixed-resolution models
    (EfficientViT/EfficientFormer) are built at IMG_SIZE so their attention
    biases match. Writes a per-model log to logs/<model>.log, early-stops on
    stalled val top-1, and saves the best weights to checkpoints/best_<model>.pt.
    """
    n_epochs = epochs if epochs is not None else _epochs
    family = get_model_info(model_name).family
    # MobileViT supports extra regularisation; pass drop_path only there.
    model_kwargs = {"drop_path": 0.1} if family == "mobilevit" else {}
    model_kwargs.update(overrides)

    model = create_model(model_name, num_classes=NUM_CLASSES, in_channels=3,
                         image_size=IMG_SIZE, **model_kwargs)
    params_m = count_parameters_millions(model)
    print(f"\n===== {model_name}  ({params_m:.2f}M params, family={family}) =====")

    optimizer = build_optimizer(model, name=OPTIMIZER, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = build_scheduler(optimizer, SCHEDULER, epochs=n_epochs,
                                steps_per_epoch=len(train_loader), lr=LR, min_lr=MIN_LR,
                                warmup_epochs=WARMUP_EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    ckpt = CKPT_DIR / f"best_{model_name}.pt"

    # Per-model file log: attach a FileHandler to the root logger for this run
    # (the Trainer logs via a child logger that propagates to root), then remove
    # it afterwards so each model gets its own clean logs/<model>.log.
    log_path = LOG_DIR / f"{model_name}.log"
    file_handler = logging.FileHandler(log_path, mode="w", encoding="utf-8")
    file_handler.setFormatter(_LOG_FMT)
    root_logger = logging.getLogger()
    root_logger.addHandler(file_handler)

    t0 = time.perf_counter()
    try:
        trainer = Trainer(
            model=model, train_loader=train_loader, val_loader=val_loader,
            optimizer=optimizer, num_classes=NUM_CLASSES, criterion=criterion,
            scheduler=scheduler, device=DEVICE, monitor=MONITOR, amp=AMP,
            grad_clip_norm=GRAD_CLIP_NORM, checkpoint_path=ckpt,
            early_stopping_patience=(EARLY_STOPPING_PATIENCE or None),
            topk=(1, 5),  # <-- top-1 AND top-5 are tracked every epoch
        )
        history = trainer.train(epochs=n_epochs)
    finally:
        root_logger.removeHandler(file_handler)
        file_handler.close()
    minutes = (time.perf_counter() - t0) / 60.0

    # Report the top-1 and top-5 of the epoch with the best validation top-1.
    best = max(history, key=lambda r: r.get("val_top1", 0.0))
    summary = {
        "params_M": round(params_m, 2),
        "best_top1": round(best.get("val_top1", 0.0), 2),
        "best_top5": round(best.get("val_top5", 0.0), 2),
        "final_top1": round(history[-1].get("val_top1", 0.0), 2),
        "final_top5": round(history[-1].get("val_top5", 0.0), 2),
        "epochs_run": len(history),
        "minutes": round(minutes, 1),
        "img_per_s": round(history[0].get("images_per_sec", 0.0), 1),
        "ckpt": str(ckpt),
        "log": str(log_path),
    }
    RESULTS[model_name] = summary
    print(f"---- {model_name}: best val top-1 {summary['best_top1']:.2f} | "
          f"top-5 {summary['best_top5']:.2f} | {summary['epochs_run']} epochs | "
          f"{summary['minutes']:.1f} min | ~{summary['img_per_s']:.0f} img/s")
    print(f"     log -> {log_path} | ckpt -> {ckpt}")
    return summary


## 7. Train each model (one cell each)

Run the cells you want — each is independent, writes its own `logs/<model>.log`,
early-stops if val top-1 stalls for `EARLY_STOPPING_PATIENCE` (10) epochs, and
saves `checkpoints/best_<model>.pt`. Tip: run **MobileViT-XXS** first as a sanity
check (smallest), or set `QUICK_TEST = True` in the config cell and re-run cells
5–6 for a fast end-to-end dry run. Each cell prints **best validation top-1 and
top-5**.

In [ ]:
# MobileViT-XXS (~1.3M)
run("mobilevit_xxs")


In [ ]:
# MobileViT-XS (~2.3M)
run("mobilevit_xs")


In [ ]:
# MobileViT-S (~5.6M)
run("mobilevit_s")


In [ ]:
# EfficientViT-M0 (~2.3M)
run("efficientvit_m0")


In [ ]:
# EfficientViT-M1 (~3.0M)
run("efficientvit_m1")


In [ ]:
# EfficientViT-M2 (~4.2M)
run("efficientvit_m2")


In [ ]:
# EfficientViT-M3 (~6.9M)
run("efficientvit_m3")


In [ ]:
# EfficientViT-M4 (~8.8M)
run("efficientvit_m4")


In [ ]:
# EfficientViT-M5 (~12.5M)
run("efficientvit_m5")


In [ ]:
# EfficientFormer-L1 (~11M)
run("efficientformer_l1")


In [ ]:
# EfficientFormer-L3 (~29M)
run("efficientformer_l3")


In [ ]:
# EfficientFormer-L7 (~66M; lower BATCH_SIZE if OOM)
run("efficientformer_l7")


In [ ]:
# EfficientFormer-L3-mini (~18M)
run("efficientformer_l3_mini")


In [ ]:
# EfficientFormer-L7-mini (~36M)
run("efficientformer_l7_mini")


In [ ]:
# --- 8. Summary table of all trained models ------------------------------------
if RESULTS:
    cols = ["params_M", "best_top1", "best_top5", "final_top1", "final_top5", "epochs_run", "minutes", "img_per_s"]
    print(f"{'model':<26}" + "".join(f"{c:>11}" for c in cols))
    print("-" * (26 + 11 * len(cols)))
    for name, r in RESULTS.items():
        print(f"{name:<26}" + "".join(f"{r[c]:>11}" for c in cols))
else:
    print("No results yet — run some model cells above.")


## 9. Inference with a trained checkpoint

```python
from PIL import Image
from inference import Predictor

predictor = Predictor.from_checkpoint(
    "checkpoints/best_mobilevit_xs.pt", model_name="mobilevit_xs",
    num_classes=NUM_CLASSES, image_size=IMG_SIZE, class_names=CLASS_NAMES,
)
probs, idx = predictor.predict(Image.open("some.jpg").convert("RGB"), topk=5)
for p, i in zip(probs.tolist(), idx.tolist()):
    print(f"{predictor.class_name(i):>22}  {p:.3f}")
```
